# NB1 · Reaching the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What the workshop does

Across six notebooks you will write a working clinical decision support system. You will
not write the code. Each step gives you a prompt; you pass it to a generative AI tool
(ChatGPT, Claude, Gemini), paste the code it returns into the blank cell and run it.

| Notebook | Layer added |
|---|---|
| NB1 | Reaching the data |
| NB2 | Cleaning, train and test split, preparation for the model |
| NB3 | Training, prediction and evaluation |
| NB4 | Explaining the model's decision |
| NB5 | Safety checks and the compliance report |
| NB6 | The web interface |

A few questions follow each step. Answer them by reading your code and its output. Those
questions are the substance of the workshop: generated code looks right at first glance,
and so do the places where it is wrong.


## Code carries forward

The first line of each paste cell reads `#@cdss step_name`. **Do not delete it.** Paste
your code below it. At the end of the notebook `kit.export()` returns every marked cell as
one block, which you paste at the head of the next notebook.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Choosing a dataset

Six datasets are prepared. All are openly accessible and reached from inside Colab. Choose
one; the later steps follow your choice.

| Code | Type | Data | Condition to predict |
|---|---|---|---|
| `mimic-icu` | table | MIMIC-IV demo, intensive care records of 100 patients | A stay exceeding three days |
| `wisconsin` | table | Breast Cancer Wisconsin, 569 samples | The mass being malignant |
| `pneumonia-mnist` | image | PneumoniaMNIST, 5,856 chest radiographs | Pneumonia present |
| `breast-mnist` | image | BreastMNIST, 780 ultrasound images | The mass being malignant |
| `mimic-ecg` | signal | MIMIC-IV-ECG demo, 659 ECGs from 92 patients | A stay exceeding three days |
| `synthetic-notes` | text | Generated clinical notes | A condition you define |

Three points are worth knowing before you choose. In the MIMIC intensive care and ECG sets
a patient has several records, so the train and test split has to be made at patient
level. The MedMNIST sets carry no patient identifier. The ECG set covers the same 92
patients as the clinical demo, so that route predicts the same outcome from the signal.


---

## Step 1 · Preparation

A program starts by importing libraries and defining the values that will not change.
Those values describe the problem and every later step reads them.

Copy the prompt for the dataset you chose and pass it to the AI tool.


### Prompt · `mimic-icu`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'mimic-icu'
PROBLEM = 'Predicting whether a patient admitted to intensive care will stay longer than three days'
DECISION_MOMENT = 'Six hours after admission to intensive care'
DATA_ROOT = 'https://physionet.org/files/mimic-iv-demo/2.2'
RANDOM_SEED = 42
DECISION_WINDOW_HOURS = 6
TARGET_THRESHOLD_DAYS = 3

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


### Prompt · `wisconsin`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'wisconsin'
PROBLEM = 'Predicting whether a breast mass is malignant'
DECISION_MOMENT = 'When the fine needle aspiration measurements are available'
DATA_ROOT = 'scikit-learn'
RANDOM_SEED = 42

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


### Prompt · `pneumonia-mnist`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'pneumonia-mnist'
PROBLEM = 'Predicting whether a paediatric chest radiograph shows pneumonia'
DECISION_MOMENT = 'Immediately after the radiograph is taken, before expert review'
DATA_ROOT = 'medmnist'
RANDOM_SEED = 42

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


### Prompt · `breast-mnist`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'breast-mnist'
PROBLEM = 'Predicting whether a breast mass seen on ultrasound is malignant'
DECISION_MOMENT = 'When the ultrasound image is acquired'
DATA_ROOT = 'medmnist'
RANDOM_SEED = 42

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


### Prompt · `mimic-ecg`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'mimic-ecg'
PROBLEM = 'Predicting from the ECG whether an intensive care stay will exceed three days'
DECISION_MOMENT = 'When the ECG is recorded'
DATA_ROOT = 'https://physionet.org/files/mimic-iv-ecg-demo/0.1'
RANDOM_SEED = 42
TARGET_THRESHOLD_DAYS = 3

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


### Prompt · `synthetic-notes`

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section.

Import the libraries needed for handling data and for machine learning. Then define
these variables:

DATASET = 'synthetic-notes'
PROBLEM = '[write the clinical problem you want to solve in one sentence]'
DECISION_MOMENT = '[when the system produces output]'
DATA_ROOT = 'synthetic'
RANDOM_SEED = 42

Print the library versions and the variables you defined. Add short comments to the
code lines.
```


In [ ]:
#@cdss hazirlik
# Paste the generated code below this line.


### Look at your code

- Which libraries were imported? Can you say what each is for?
- What would change if the random seed were not fixed?
- Are the variables defined at the top of the file rather than inside a function? Why does
  keeping threshold values in one place matter in a clinical system?


---

## Step 2 · Loading the data

Now you will have the loading function written. Every prompt produces a result of the same
shape: a dataframe with `patient_id` and `target`. That shared shape is what lets the later
steps proceed regardless of data type.

File addresses and column names are written into the prompts. The AI tool does not know
them; it guesses what it does not know, and the guess is usually wrong.


### Prompt · `mimic-icu`

```
Using the variables above, write a function that loads the data. Name it load_data.

Three files are needed, all gzipped CSV:
  {DATA_ROOT}/icu/icustays.csv.gz     subject_id, hadm_id, stay_id, first_careunit,
                                      intime, outtime, los
  {DATA_ROOT}/hosp/patients.csv.gz    subject_id, gender, anchor_age
  {DATA_ROOT}/hosp/admissions.csv.gz  subject_id, hadm_id, admission_type, insurance

Start from the intensive care stays. Drop stays shorter than DECISION_WINDOW_HOURS hours.
Create the target column: 1 where los exceeds TARGET_THRESHOLD_DAYS, 0 otherwise. Rename
subject_id to patient_id. Add the demographic columns from the other two files; the row
count must not change.

Drop los and outtime from the result. Both are known only after the patient is discharged
and are not available at the decision moment.

Call the function, keep the result in cohort, show the first five rows.
```


### Prompt · `wisconsin`

```
Write a function that loads the Breast Cancer Wisconsin dataset. Name it load_data. The
set ships with scikit-learn and opens with load_breast_cancer.

Turn it into a dataframe. Create the target column: 1 where the mass is malignant, 0 where
benign. Each row is a separate person, so derive patient_id from the row order and note in
a comment that it is not a real patient identifier.

Call the function, keep the result in cohort, show the first five rows.
```


### Prompt · `pneumonia-mnist` and `breast-mnist`

```
Write a function that loads an image set from MedMNIST. Name it load_data.
Install with: pip install medmnist

Use PneumoniaMNIST where DATASET is 'pneumonia-mnist' and BreastMNIST where it is
'breast-mnist'. The images are 28x28 and single channel. The set arrives split into train,
validation and test; combine all three, we will split it ourselves in NB2.

Build a dataframe with one row per image: image and target columns. This set has no
patient identifier, so derive patient_id from the row order and note in a comment that it
is not a real one. If the set is large, limit it to the first 2000 images.

Call the function, keep the result in cohort, print the class distribution.
```


### Prompt · `mimic-ecg`

```
Write a function that loads the ECG recordings and takes the target from the clinical
record. Name it load_data. Install with: pip install wfdb

The recordings are at {DATA_ROOT}, in WFDB format, ten seconds long at 500 Hz. The record
list is in record_list.csv. Each patient's recordings sit in a folder named after the
patient identifier.

Read record_list.csv and take the first 200 records. Read each with wfdb and keep the
first lead only.

Take the target from the clinical record at
  https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz  (subject_id, los)
Find the longest stay for each patient; 1 where it exceeds TARGET_THRESHOLD_DAYS, 0
otherwise. Drop patients with no intensive care stay.

The result must hold patient_id, signal and target. Print how many patients have more than
one recording. Call the function and keep the result in cohort.
```


### Prompt · `synthetic-notes`

```
Write a function that generates synthetic clinical notes. Name it load_data.

Give the notes the features that make real clinical notes hard: abbreviations, negation,
expressions of uncertainty, repeated templated sentences and sections copied forward from
earlier notes. Generate them for the condition described in PROBLEM and DECISION_MOMENT.
The condition must not be recoverable from a single word.

The result must hold patient_id, note and target, with several notes per patient. Call the
function, keep the result in cohort, print three example notes.
```


In [ ]:
#@cdss veri_yukleme
# Paste the generated code below this line.


### Look at the data

Run the supplied cell below unchanged.


In [ ]:
print('Rows      :', len(cohort))
print('Columns   :', cohort.shape[1])
print('Patients  :', cohort['patient_id'].nunique())
print('Event rate:', f"{cohort['target'].mean():.1%}")
print()
print('Columns:', list(cohort.columns))
print()
print('Five columns with the most missing data:')
print(cohort.isna().mean().sort_values(ascending=False).head(5).round(3).to_string())


### Look at your code

- Is the patient count lower than the row count? Consider what that means for the train
  and test split; NB2 returns to this.
- What is the event rate? If it is imbalanced, which performance measure becomes
  misleading?
- Which columns were deliberately dropped? Can you explain why?
- Is there any remaining column that would not be available at the decision moment? If so,
  correct the prompt and regenerate.

The last question is the important one. Information not present at the decision moment
makes the model look almost perfect and useless in the field. This is called data leakage;
it raises no error and proceeds silently.


---

## End of notebook · Collect the code

The cell below returns what you wrote here as one block. Copy it into the first cell of
NB2. The cell also downloads the file to your computer; if the download does not start,
take it from the file panel on the left.


In [ ]:
code_so_far = kit.export('cdss_nb1.py')


## What this notebook did

The system now prepares itself and loads the data. Two habits were formed: what the data
contains was stated to the tool explicitly, and information unavailable at the decision
moment was removed during loading.

NB2 adds cleaning, the train and test split, and preparation for the model.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution.
